# Wireless Coverage Lakeflow — Pipeline Validation

Checks the terminal output tables of the Wireless Coverage Lakeflow pipeline:
silver surface rasters, gold H3 elevation grids, and the siting outputs (quick-pass
screen, candidate sites, coverage depth, and the top-10 site coverage footprint).

For each table the notebook verifies that it exists, is non-empty, and carries its
expected columns. The five H3 grid tables (`wc_h3_dem_res10`, `wc_h3_dsm_res10`,
`wc_h3_chm_res10`, `wc_h3_coverage_res12`, `wc_h3_coverage_top10_res12`) are
additionally checked for their `parent_cellid` / `parent_cellid_res` stable join key.
Coverage depth is verified to be at least 1 line-of-sight count per covered cell.
The notebook also reports how many of the `.laz` files `laz_inventory` lists have points in
`wc_lidar_pts`, because `read_paths(..., on_error="skip")` skips an unreadable file with only
a warning in the executor logs.

Static map previews of the coverage depth (H3 res 12) and the canopy height model
(H3 res 10) are rendered inline so the results commit as images in the repository.

**Run as the final `validate` task in the Wireless Coverage Lakeflow job.** The
notebook raises loudly on any accumulated assertion failure so the job task reports
a clear error; it prints `VALIDATION PASS` when all checks succeed.

---
**Version:** 0.5.3

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
# Resolve catalog and schema from job task parameters (dbutils.widgets).
# Defaults match the Lakeflow job variables so a manual run works without setup.
dbutils.widgets.text("catalog", "geospatial_docs")
dbutils.widgets.text("schema",  "wireless_coverage_lf")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")


def _fqn(name):
    """Backtick-quoted CATALOG.SCHEMA.name (hyphens OK; embedded backticks doubled)."""
    return ".".join("`" + p.replace("`", "``") + "`" for p in (CATALOG, SCHEMA, name))


print(f"Validating  {CATALOG}.{SCHEMA}")

In [ ]:
import json

from pyspark.sql import functions as F

# ── Terminal tables and their required columns ────────────────────────────────
# Each entry: (table_name, [required_column, ...]).
# H3 grid tables that carry the stable parent join key are annotated accordingly.
_TABLES = [
    ("wc_surface_dsm",             ["dsm", "tx", "ty"]),
    ("wc_surface_dtm",             ["dtm", "tx", "ty"]),
    ("wc_surface_chm",             ["chm", "tx", "ty"]),
    # Gold H3 elevation grids — must have the res-join_res parent join key and h3 string columns.
    ("wc_h3_dem_res10",            ["cellid", "h3", "res", "band_level", "elev_lo", "elev_hi",
                                    "parent_cellid", "parent_cellid_res"]),
    ("wc_h3_dsm_res10",            ["cellid", "h3", "res", "band_level", "elev_lo", "elev_hi",
                                    "parent_cellid", "parent_cellid_res"]),
    ("wc_h3_chm_res10",            ["cellid", "h3", "res", "max_chm_z",
                                    "parent_cellid", "parent_cellid_res"]),
    # Siting intermediates (H3 surface / ground bins at res 13) — have h3 string columns and clustering.
    ("wc_h3_surf_r13",             ["cellid", "h3", "z"]),
    ("wc_h3_grnd_r13",             ["cellid", "h3", "z"]),
    # Siting outputs.
    ("wc_h3_quickpass_res9",       ["lon", "lat", "quick_frac"]),
    ("wc_h3_candidate_sites_res9", ["cellid", "h3", "lon", "lat", "required_mast",
                                    "viewshed_cells", "viewshed_pct", "observer_z"]),
    # Coverage depth — must have the parent join key and h3 string column.
    ("wc_h3_coverage_res12",       ["cellid", "h3", "tower_los_count", "res",
                                    "parent_cellid", "parent_cellid_res"]),
    ("wc_h3_coverage_top10_res12", ["cellid", "h3", "tower_los_count", "res",
                                    "parent_cellid", "parent_cellid_res"]),
    # Per-site viewshed: one row per sited tower + visible cell pair (res 12).
    ("wc_h3_site_viewshed_res12",  ["site_cellid", "site_h3", "cellid", "h3", "res",
                                    "parent_cellid", "parent_cellid_res"]),
]

# Tables that should be liquid-clustered. Read from the table properties, which works for
# both the siting Delta tables and the pipeline's materialized views (DESCRIBE DETAIL rejects MVs).
_CLUSTERED_TABLES = {
    "wc_h3_dem_res10": ["cellid"],
    "wc_h3_dsm_res10": ["cellid"],
    "wc_h3_chm_res10": ["cellid"],
    "wc_h3_surf_r13": ["cellid"],
    "wc_h3_grnd_r13": ["cellid"],
    "wc_h3_candidate_sites_res9": ["cellid"],
    "wc_h3_coverage_res12": ["cellid"],
    "wc_h3_coverage_top10_res12": ["cellid"],
    "wc_h3_site_viewshed_res12": ["site_cellid", "cellid"],
}

results = []

for _tbl, _required in _TABLES:
    _full = _fqn(_tbl)
    try:
        if not spark.catalog.tableExists(_full):
            results.append((_tbl, "FAIL", f"table not found: {_full}"))
            continue
        _df = spark.table(_full)
        _cols = set(_df.columns)
        _missing = [c for c in _required if c not in _cols]
        if _missing:
            results.append((_tbl, "FAIL", f"missing columns: {_missing}"))
            continue
        _n = _df.count()
        if _n == 0:
            results.append((_tbl, "FAIL", "table is empty (count = 0)"))
        else:
            results.append((_tbl, "PASS", f"{_n:,} rows"))
    except Exception as _e:
        results.append((_tbl, "FAIL", str(_e)))

# Check clustering on tables that should be clustered
for _tbl, _expected_cluster_cols in _CLUSTERED_TABLES.items():
    try:
        _props = {r["key"]: r["value"] for r in spark.sql(f"SHOW TBLPROPERTIES {_fqn(_tbl)}").collect()}
        _got = [c for path in json.loads(_props.get("clusteringColumns", "[]")) for c in path]
        _ok = _got == _expected_cluster_cols
        results.append((_tbl + " (clustering)", "PASS" if _ok else "FAIL",
                        f"cluster_by={_got}" + ("" if _ok else f", expected {_expected_cluster_cols}")))
    except Exception as _e:
        results.append((_tbl + " (clustering)", "FAIL", str(_e)))

# Print summary table
_max_w = max(len(t) for t, _, _ in results)
for _tbl, _status, _detail in results:
    _tag = f"[{_status}]"
    print(f"  {_tag:<7}  {_tbl:<{_max_w}}  {_detail}")


In [ ]:
# ── Coverage-depth sanity ─────────────────────────────────────────────────────
# Every covered cell must have at least one tower with line of sight.
# Print the distribution (up to 20 depth buckets) as a quick visual check.
_cov_full = _fqn("wc_h3_coverage_res12")
try:
    _cov = spark.table(_cov_full)
    _zero = _cov.where(F.col("tower_los_count") < 1).count()
    if _zero > 0:
        results.append(("wc_h3_coverage_res12 (depth)", "FAIL",
                        f"{_zero:,} cells with tower_los_count < 1"))
    else:
        print("Coverage depth: all cells have tower_los_count >= 1")

    _dist = (
        _cov.groupBy("tower_los_count")
        .count()
        .orderBy("tower_los_count")
        .limit(20)
        .collect()
    )
    _max_count = max((r["count"] for r in _dist), default=1)
    print("Depth distribution (tower_los_count -> cell count):")
    for _r in _dist:
        _bar = "#" * max(1, round(_r["count"] / _max_count * 40))
        print(f"  {_r['tower_los_count']:>5}  {_r['count']:>10,}  {_bar}")
except Exception as _e:
    results.append(("wc_h3_coverage_res12 (depth)", "FAIL", str(_e)))

In [ ]:
# ── Silver input coverage ─────────────────────────────────────────────────────
# wc_lidar_pts reads the files laz_inventory lists with read_paths(..., on_error="skip"),
# so a file that cannot be decoded is skipped with only a warning in the executor logs.
# Compare the distinct source files in wc_lidar_pts with the inventoried files. A file
# with no point inside the AOI also adds no row, so the gap is reported, not failed.
try:
    _n_inv = spark.table(_fqn("laz_inventory")).select(F.countDistinct("path")).first()[0]
    _n_read = spark.table(_fqn("wc_lidar_pts")).select(F.countDistinct("_source_path")).first()[0]
    print(f"Input coverage: {_n_read:,} of {_n_inv:,} inventoried .laz files have points in wc_lidar_pts")
    print(f"  {_n_inv - _n_read:,} without points: skipped as unreadable, or no point inside the AOI")
except Exception as _e:
    results.append(("wc_lidar_pts (input coverage)", "FAIL", str(_e)))

In [ ]:
# ── Static map previews ───────────────────────────────────────────────────────
# Vizx static renders commit as PNG images in the notebook and render on GitHub.
# The vizx import is deferred so this cell skips gracefully when the extra is absent.
try:
    from databricks.labs.gbx.vizx import cells_as_gdf, plot_raster, plot_static, vector_layer

    # Raster surface preview: one CHM tile rendered with plot_raster so that an
    # actual surface raster (not just the H3-aggregated form) is visible inline.
    _chm_tbl = _fqn("wc_surface_chm")
    _tile_row = spark.table(_chm_tbl).limit(1).first()
    if _tile_row:
        plot_raster(_tile_row["chm"]["raster"], cmap="viridis", fig_w=7, fig_h=7)

    # Coverage depth: each H3 cell colored by the number of towers with line of sight.
    _cov_df = (
        spark.table(_fqn("wc_h3_coverage_res12"))
        .select("cellid", "tower_los_count")
    )
    _cov_gdf = cells_as_gdf(_cov_df, "cellid",
                             extra_cols=("tower_los_count",), max_rows=150_000)
    plot_static(
        [vector_layer(_cov_gdf, geom_col="geometry", column="tower_los_count",
                      cmap="viridis", opacity=0.8, label="LOS depth (towers)")],
        title="Coverage depth \u2014 tower line-of-sight count per H3 cell (res 12)",
    )

    # Canopy height model: max structure / canopy height per H3 cell.
    _chm_h3_df = (
        spark.table(_fqn("wc_h3_chm_res10"))
        .select("cellid", "max_chm_z")
    )
    _chm_gdf = cells_as_gdf(_chm_h3_df, "cellid",
                             extra_cols=("max_chm_z",), max_rows=150_000)
    plot_static(
        [vector_layer(_chm_gdf, geom_col="geometry", column="max_chm_z",
                      cmap="YlGn", opacity=0.8, label="Max canopy height (m)")],
        title="Canopy height model \u2014 max structure height per H3 cell (res 10)",
    )
except ImportError:
    print("vizx extra not available; skipping static previews")

In [ ]:
# ── Final assertion ───────────────────────────────────────────────────────────
# Raise if any check above collected a FAIL so the job task reports a clear error.
_failures = [(t, d) for t, s, d in results if s == "FAIL"]
if _failures:
    _msg = "\n".join(f"  {t}: {d}" for t, d in _failures)
    raise AssertionError(
        f"Validation failures ({len(_failures)} of {len(results)}):\n{_msg}"
    )
print("VALIDATION PASS")